<a href="https://colab.research.google.com/github/Tamashacker/GoogleColab_ComfyUI/blob/main/comfyui_colab_Flux.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the [ComfyUI](https://github.com/comfyanonymous/ComfyUI) repo and install the requirements.

In [1]:
!git clone -q https://github.com/comfyanonymous/ComfyUI
%pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu130
%pip install -q -r ./ComfyUI/requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.6/794.6 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.8/92.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.0

In [2]:
# Some Helpful Custom Nodes
!git clone -q https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/comfyui-manager

!git clone -q https://github.com/ClownsharkBatwing/RES4LYF.git /content/ComfyUI/custom_nodes/RES4LYF
!git clone -q https://github.com/rgthree/rgthree-comfy.git /content/ComfyUI/custom_nodes/rgthree-comfy
!git clone -q https://github.com/yolain/ComfyUI-Easy-Use.git /content/ComfyUI/custom_nodes/ComfyUI-Easy-Use

Download the vae and dual text encoder needed for FLUX - (Compulsory)

In [3]:
!wget -q --show-progress https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors -P /content/ComfyUI/models/text_encoders/
!wget -q --show-progress https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors -P /content/ComfyUI/models/text_encoders/
!wget -q --show-progress https://huggingface.co/cocktailpeanut/xulf-dev/resolve/main/ae.safetensors -P /content/ComfyUI/models/vae/
#Get diffusion model
!wget -q --show-progress https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-dev-fp8.safetensors -P /content/ComfyUI/models/diffusion_models/

#Get other model
!wget -q --show-progress https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-dev-fp8.safetensors -P /content/ComfyUI/models/diffusion_models/

clip_l.safetensors  100%[===================>] 234.74M   230MB/s    in 1.0s    
t5xxl_fp8_e4m3fn.sa 100%[===================>]   4.56G   176MB/s    in 31s     
ae.safetensors      100%[===================>] 319.77M   319MB/s    in 1.0s    



The **FP8 version** runs smoothly on the free tier (without LoRA), but if you want to use it with a LoRA, it requires more resources than the free Colab tier provides.  
In that case, you can opt for the [gguf](https://huggingface.co/collections/QuantStack/flux-ggufs-68264cfc663d50c418940b30) quantized versions or the [Nunchaku](https://huggingface.co/nunchaku-tech/models) versions instead.

>**Note** : You need only one , either a gguf quantized model (under 10GB) or the Nunchanku model

---


### Option 1 : fp8 quantized - from [Comfy-Org](https://huggingface.co/Comfy-Org/flux1-dev/tree/main)

In [4]:
# fp8 - needs more than free tier
# !wget -q --show-progress https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors -P /content/ComfyUI/models/checkpoints

### Run ComfyUI with ngrok

In [10]:
!pip install -q pyngrok

from pyngrok import ngrok
import subprocess, socket, time
from google.colab import userdata

NGROK_TOKEN = userdata.get("NGROK_TOKEN")

# Set ngrok token
!ngrok config add-authtoken $NGROK_TOKEN

# Start ComfyUI in background
subprocess.Popen(["python", "/content/ComfyUI/main.py", "--dont-print-server"])

# Wait until port is open
port = 8188
while True:
    try:
        sock = socket.create_connection(("127.0.0.1", port), timeout=2)
        sock.close()
        print("✅ ComfyUI server is running on port", port)
        break
    except OSError:
        print("⏳ Waiting for ComfyUI to start...")
        time.sleep(2)

# Start ngrok tunnel
public_url = ngrok.connect(8188, bind_tls=True)
print("🌐 Public URL:", public_url)


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
⏳ Waiting for ComfyUI to start...
✅ ComfyUI server is running on port 8188
🌐 Public URL: NgrokTunnel: "https://marlana-couped-northerly.ngrok-free.dev" -> "http://localhost:8188"
